# Notebook 3: GSPO 强化学习训练 + GGUF 量化导出

本 Notebook 完成以下工作:
1. 加载 SFT 模型 + LoRA Adapter
2. 加载 RL 训练数据 (prompt+answer 对)
3. 配置 GSPO / GRPO 算法
4. 实现 Rule-based Reward Functions
5. 执行 RL 训练 (4× A5000 多卡)
6. 导出 GGUF 量化模型 (Q8_0, Q4_K_M)

**GSPO = GRPO + importance_sampling_level="sequence" + loss_type="dr_grpo"**

**核心创新**:
- 无需 Reward Model（使用 rule-based reward）
- 无需 Critic Model（GRPO 使用 group-relative advantage）
- Diffusion RL loss (dr_grpo) 提供更稳定的梯度

**硬件需求**: 4× A5000 (24GB each)
**预计耗时**: 4-8 小时 (2000 prompts, G=8, 1 epoch)

In [ ]:
# ============================================================
# 0. 环境初始化
# ============================================================

import os
import sys
import json
from pathlib import Path
import yaml
import torch
from datasets import Dataset

# 项目根目录
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from utils.rewards import (
    format_reward,
    anomaly_reward,
    correctness_reward,
    compute_rewards,
    extract_answer,
)
from utils.schema import RLDataFormatter

print(f" Project Root: {PROJECT_ROOT}")
print(f" CUDA devices: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GB)")

In [ ]:
# ============================================================
# 1. 加载配置
# ============================================================

with open(PROJECT_ROOT / "config" / "gspo.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)

print("=== GSPO Configuration ===")
for key, val in CONFIG.items():
    if isinstance(val, dict):
        print(f"  {key}:")
        for k, v in val.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {val}")

In [ ]:
# ============================================================
# 2. 加载 SFT 模型
# ============================================================

from unsloth import FastLanguageModel
from peft import PeftModel

model_cfg = CONFIG["model"]
grpo_cfg = CONFIG["grpo"]

# 加载基础 4-bit 模型
print(f" Loading base model: {model_cfg['base_model']}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_cfg["base_model"],
    max_seq_length=model_cfg["max_completion_length"] + model_cfg["max_prompt_length"],
    dtype=None,
    load_in_4bit=True,
)

print(f" Base model loaded!")

# 加载 SFT LoRA Adapter (从 Notebook 2 产出)
ADAPTER_PATH = Path(model_cfg["adapter_path"]) / "lora_adapter"

if ADAPTER_PATH.exists():
    print(f"\n Loading SFT LoRA adapter from: {ADAPTER_PATH}")
    model = PeftModel.from_pretrained(model, str(ADAPTER_PATH))
    print("   ✅ Adapter loaded!")
else:
    print(f"\n ⚠️ SFT adapter not found at {ADAPTER_PATH}")
    print("   Using base model without SFT adapter (demo mode).")
    print("   For best results, run Notebook 2 first.")

# 确保 tokenizer 有 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"\n Tokenizer:")
print(f"  pad_token: {tokenizer.pad_token} ({tokenizer.pad_token_id})")
print(f"  eos_token: {tokenizer.eos_token} ({tokenizer.eos_token_id})")
print(f"  chat_template: {'✅' if tokenizer.chat_template else '❌'}")

In [ ]:
# ============================================================
# 3. 加载 RL 训练数据
# ============================================================

RL_DATA_PATH = PROJECT_ROOT / "data" / "rl_train.jsonl"

if RL_DATA_PATH.exists():
    print(f" Loading RL data from: {RL_DATA_PATH}")
    with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
        rl_data = [json.loads(line) for line in f if line.strip()]
    
    # 限制样本数
    max_samples = CONFIG["dataset"].get("max_samples", 2000)
    if len(rl_data) > max_samples:
        import random
        random.seed(3407)
        rl_data = random.sample(rl_data, max_samples)
    
    dataset = Dataset.from_list(rl_data)
    print(f"   Loaded {len(dataset)} samples")
else:
    print(f" ⚠️ RL data not found at {RL_DATA_PATH}")
    print("   Creating demo data...")
    
    demo_data = [
        {
            "prompt": "<|im_start|>system\nYou are a math tutor. Think step by step.\n<|im_end|>\n<|im_start|>user\nWhat is 15 + 27?\n<|im_end|>\n<|im_start|>assistant\n",
            "answer": "42"
        },
        {
            "prompt": "<|im_start|>system\nYou are a math tutor. Think step by step.\n<|im_end|>\n<|im_start|>user\nIf x + 5 = 12, what is x?\n<|im_end|>\n<|im_start|>assistant\n",
            "answer": "7"
        },
        {
            "prompt": "<|im_start|>system\nYou are a math tutor. Think step by step.\n<|im_end|>\n<|im_start|>user\nCalculate the area of a circle with radius 3.\n<|im_end|>\n<|im_start|>assistant\n",
            "answer": "28.27"
        },
        {
            "prompt": "<|im_start|>system\nYou are a math tutor. Think step by step.\n<|im_end|>\n<|im_start|>user\nWhat is the square root of 144?\n<|im_end|>\n<|im_start|>assistant\n",
            "answer": "12"
        },
        {
            "prompt": "<|im_start|>system\nYou are a math tutor. Think step by step.\n<|im_end|>\n<|im_start|>user\nSolve: 3x - 7 = 14\n<|im_end|>\n<|im_start|>assistant\n",
            "answer": "7"
        },
    ] * 10  # 复制 10 次获得 50 条 demo 数据
    
    dataset = Dataset.from_list(demo_data)
    print(f"   Created {len(dataset)} demo samples")

# 预览
print(f"\n=== Sample 0 ===")
print(f"Prompt: {dataset[0]['prompt'][:100]}...")
print(f"Answer: {dataset[0]['answer'][:100]}...")

In [ ]:
# ============================================================
# 4. 定义 Reward Functions
# ============================================================
#
# 关键设计决策:
# - 使用 rule-based reward (无需训练 Reward Model)
# - 三个维度的奖励: 格式 / 异常 / 正确性
# - 组合权重由 config 控制
#
# ============================================================

reward_cfg = CONFIG["rewards"]

def combined_reward_function(prompts, completions, answer, **kwargs):
    """
    GRPOTrainer 兼容的组合奖励函数。
    
    Args:
        prompts:     原始 prompt 列表
        completions: 模型生成的 completion 列表
        answer:      正确答案列表 (从 dataset 中提取)
    
    Returns:
        rewards: float 列表, 每条 completion 的总分
    """
    # 提取权重
    weights = {}
    for fn_cfg in reward_cfg["functions"]:
        weights[fn_cfg["name"].replace("_reward", "")] = fn_cfg["weight"]
    
    match_type = reward_cfg["correctness"].get("match_type", "auto")
    
    # 计算综合奖励
    rewards = compute_rewards(
        completions=list(completions),
        solution=list(answer),
        weights=weights,
        match_type=match_type,
    )
    
    return rewards


# 测试 reward function
print("=== Reward Function Test ===")
test_prompts = ["test"] * 3
test_completions = [
    " thinkingLet's solve this response\n<answer>42</answer>",
    "The answer is 42",
    " thinkingHmm response<answer>99</answer><answer>100</answer>",
]
test_answers = ["42", "42", "42"]

rewards = combined_reward_function(test_prompts, test_completions, test_answers)
for i, (c, r) in enumerate(zip(test_completions, rewards)):
    print(f"  [{i}] reward={r:.2f} | {c[:60]}...")
print("\n Reward function is working correctly!")

In [ ]:
# ============================================================
# 5. 配置 GRPOTrainer (GSPO 模式)
# ============================================================

from trl import GRPOTrainer, GRPOConfig

# GRPO 训练参数
training_args = GRPOConfig(
    # 输出
    output_dir="./outputs/gspo_qwen",
    
    # Batch (4 GPU, 每卡 batch=1, grad_accum=4 → 16 prompts/step)
    per_device_train_batch_size=grpo_cfg["per_device_train_batch_size"],
    gradient_accumulation_steps=grpo_cfg["gradient_accumulation_steps"],
    
    # Epochs
    num_train_epochs=grpo_cfg["num_train_epochs"],
    
    # 学习率 (RL 阶段要远低于 SFT)
    learning_rate=grpo_cfg["learning_rate"],
    lr_scheduler_type=grpo_cfg["lr_scheduler_type"],
    warmup_ratio=grpo_cfg["warmup_ratio"],
    optim=grpo_cfg["optim"],
    max_grad_norm=grpo_cfg["max_grad_norm"],
    
    # GSPO 关键参数
    num_generations=grpo_cfg["num_generations"],          # G=8
    importance_sampling_level=grpo_cfg["importance_sampling_level"],  # "sequence"
    loss_type=grpo_cfg["loss_type"],                      # "dr_grpo"
    mask_truncated_completions=grpo_cfg["mask_truncated_completions"],
    
    # GRPO 超参数
    beta=grpo_cfg["beta"],
    epsilon=grpo_cfg["epsilon"],
    epsilon_high=grpo_cfg["epsilon_high"],
    temperature=grpo_cfg["temperature"],
    
    # 生成参数
    max_completion_length=model_cfg["max_completion_length"],
    max_prompt_length=model_cfg["max_prompt_length"],
    
    # 日志
    logging_steps=grpo_cfg["logging_steps"],
    save_steps=grpo_cfg["save_steps"],
    save_total_limit=grpo_cfg["save_total_limit"],
    
    # 精度
    fp16=CONFIG["hardware"]["fp16"],
    bf16=CONFIG["hardware"]["bf16"],
    
    # 报告
    report_to=CONFIG["hardware"].get("report_to", "none"),
    run_name=CONFIG["hardware"].get("run_name", "gspo_run"),
    
    # 种子
    seed=3407,
)

print("=== GRPO Training Arguments ===")
print(f"  num_generations (G): {training_args.num_generations}")
print(f"  importance_sampling_level: {training_args.importance_sampling_level}")
print(f"  loss_type: {training_args.loss_type}")
print(f"  beta (KL penalty): {training_args.beta}")
print(f"  epsilon: {training_args.epsilon}")
print(f"  epsilon_high: {training_args.epsilon_high}")
print(f"  learning_rate: {training_args.learning_rate}")
print(f"  max_completion_length: {training_args.max_completion_length}")

In [ ]:
# ============================================================
# 6. 创建 GRPOTrainer 并开始训练
# ============================================================

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[combined_reward_function],
)

print("\n" + "=" * 60)
print("  STARTING GSPO RL TRAINING")
print(f"  Dataset: {len(dataset)} prompts")
print(f"  Group size: G={training_args.num_generations}")
print(f"  Algorithm: {training_args.loss_type} + {training_args.importance_sampling_level}-level IS")
print(f"  GPUs: {torch.cuda.device_count()}")
print("=" * 60 + "\n")

# 开始训练
trainer.train()

print("\n" + "=" * 60)
print("  GSPO RL Training Complete!")
print("=" * 60)

In [ ]:
# ============================================================
# 7. 保存 RL 模型
# ============================================================

OUTPUT_DIR = Path("./outputs/gspo_qwen")
RL_FINAL = OUTPUT_DIR / "rl_final"
RL_FINAL.mkdir(parents=True, exist_ok=True)

# 保存 LoRA adapter
print(" Saving RL-tuned LoRA adapter...")
model.save_pretrained(str(RL_FINAL / "lora_adapter"))
tokenizer.save_pretrained(str(RL_FINAL / "lora_adapter"))

# 合并完整模型
print("\n Merging LoRA into base model...")
try:
    merged_model = model.merge_and_unload()
    merged_dir = RL_FINAL / "merged_model"
    merged_model.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    print(f"   ✅ Merged model saved to {merged_dir}")
    del merged_model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ⚠️ Merge failed: {e}")
    print(f"   Merge later with: model.merge_and_unload()")

print(f"\n All RL artifacts saved to: {RL_FINAL}")

In [ ]:
# ============================================================
# 8. GGUF 量化导出
# ============================================================
#
# 将合并后的 HuggingFace 模型转为 GGUF 格式,
# 支持 Q8_0 (几乎无损) 和 Q4_K_M (推荐) 两种量化级别
#
# 注意: 需要先执行上一步的 merge_and_unload()
# ============================================================

gguf_cfg = CONFIG["gguf"]

if not gguf_cfg.get("enabled", True):
    print(" GGUF export is disabled in config. Skipping.")
else:
    MERGED_PATH = RL_FINAL / "merged_model"
    GGUF_OUTPUT = Path(gguf_cfg["output_dir"])
    GGUF_OUTPUT.mkdir(parents=True, exist_ok=True)
    
    if not MERGED_PATH.exists():
        print(f" ❌ Merged model not found at {MERGED_PATH}")
        print("    Run the merge step above first.")
    else:
        print(f" Converting HF model → GGUF")
        print(f"  Source: {MERGED_PATH}")
        print(f"  Target: {GGUF_OUTPUT}")
        print(f"  Quantizations: {gguf_cfg['quantization_types']}")
        
        # 方法 1: 使用 llama.cpp 的 convert_hf_to_gguf.py
        # 需要先 clone llama.cpp
        LLAMA_CPP_DIR = PROJECT_ROOT.parent / "llama.cpp"
        
        if not LLAMA_CPP_DIR.exists():
            print("\n 📦 Cloning llama.cpp...")
            import subprocess
            subprocess.run(
                ["git", "clone", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)],
                check=True
            )
            # Build llama.cpp
            subprocess.run(["make", "-j"], cwd=str(LLAMA_CPP_DIR), check=True)
            print("   ✅ llama.cpp built!")
        
        # 步骤 1: HF → FP16 GGUF
        FP16_GGUF = GGUF_OUTPUT / "model-fp16.gguf"
        print(f"\n  Step 1: Converting to FP16 GGUF...")
        
        convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
        if convert_script.exists():
            import subprocess
            result = subprocess.run(
                [
                    sys.executable, str(convert_script),
                    str(MERGED_PATH),
                    "--outfile", str(FP16_GGUF),
                    "--outtype", "f16",
                ],
                capture_output=True,
                text=True
            )
            if result.returncode == 0:
                print(f"   ✅ FP16 GGUF: {FP16_GGUF}")
                size_mb = FP16_GGUF.stat().st_size / 1024**2
                print(f"   Size: {size_mb:.0f} MB")
            else:
                print(f"   ❌ Conversion failed:")
                print(result.stderr[-500:])
        
        # 步骤 2: 量化
        for qtype in gguf_cfg["quantization_types"]:
            q_output = GGUF_OUTPUT / f"model-{qtype}.gguf"
            print(f"\n  Step 2: Quantizing to {qtype}...")
            
            quantize_bin = LLAMA_CPP_DIR / "quantize"
            if quantize_bin.exists():
                result = subprocess.run(
                    [str(quantize_bin), str(FP16_GGUF), str(q_output), qtype],
                    capture_output=True,
                    text=True
                )
                if result.returncode == 0:
                    size_mb = q_output.stat().st_size / 1024**2
                    print(f"   ✅ {qtype}: {q_output}")
                    print(f"   Size: {size_mb:.0f} MB")
                else:
                    print(f"   ❌ Quantization failed:")
                    print(result.stderr[-300:])
            else:
                print(f"   ❌ quantize binary not found. Build llama.cpp first.")
        
        print(f"\n" + "=" * 50)
        print(f"  GGUF Export Complete!")
        print(f"  Files in {GGUF_OUTPUT}:")
        for f in sorted(GGUF_OUTPUT.glob("*.gguf")):
            size_mb = f.stat().st_size / 1024**2
            print(f"    {f.name}: {size_mb:.0f} MB")
        print("=" * 50)

In [ ]:
# ============================================================
# 9. 推理对比: RL 前后效果对比
# ============================================================

test_prompts = [
    "如果 x^2 + 6x + 9 = 0，求 x 的值。请展示推导过程。",
    "一个矩形的长是宽的两倍，周长是 36 厘米。求矩形的面积。",
    "解释为什么 0.999... = 1。",
]

print("=== Post-RL Inference Test ===\n")

FastLanguageModel.for_inference(model)

for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": "You are a helpful math tutor. Think step by step and put your final answer in <answer>...</answer> tags."},
        {"role": "user", "content": prompt},
    ]
    
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    extracted_answer = extract_answer(response)
    
    print(f"--- Test {i+1} ---")
    print(f"Prompt: {prompt[:80]}...")
    print(f"Response: {response[:250]}...")
    if extracted_answer:
        print(f"Extracted Answer: {extracted_answer}")
    print()

In [ ]:
# ============================================================
# 10. 训练总结
# ============================================================

print("=" * 60)
print("  FULL PIPELINE SUMMARY")
print("=" * 60)
print(f"""
  Pipeline: Data → SFT → GSPO → GGUF
  
  Stage 1 — Data Preparation:
    Source: R6410418/Chinese-Qwen3-235B-Thinking-2507-Distill-100k
    Source: R6410418/gpt-oss-120b-distilled-reasoning
    SFT samples: ~100K
    RL samples:  {len(dataset)}
  
  Stage 2 — SFT Training:
    Model: {model_cfg['base_model']}
    Method: QLoRA (NF4)
    LoRA rank: r=16
    Epochs: 3
  
  Stage 3 — GSPO RL Training:
    Algorithm: {grpo_cfg['loss_type']} + {grpo_cfg['importance_sampling_level']}-level IS
    Group size: G={grpo_cfg['num_generations']}
    Reward: rule-based (format + anomaly + correctness)
    GPUs: {torch.cuda.device_count()}× A5000
  
  Stage 4 — GGUF Export:
    Quantizations: {gguf_cfg['quantization_types']}
    Output: {gguf_cfg['output_dir']}
""")
print("=" * 60)
print("  ✅ Full pipeline complete!")
print("=" * 60)